# CSON V1 — Constrained Spectral Operator Network
Robust neural surrogate for Ta(k) marginal stability curves.
- Transformer encoder (5 layers, d=320, 8 heads) over 23 branch descriptors
- Spectral-normalised linear layers (Lipschitz constraint)
- Chebyshev decoder with 48 modes on k_norm in [-1, 1]
- Sigmoid clamping guarantees ta_norm in (0, 1)
- Loss = kink-weighted MSE + spectral-decay penalty + smoothness penalty
- Ensemble of 5 seeds for robustness; median at inference

In [ ]:
EPOCHS = 1000
BATCH_SIZE = 16
LR = 2e-4
N_SEEDS = 5
CTX_NOISE = 0.02
N_MODES = 48
D_MODEL = 320
NUM_LAYERS = 5
LAM_SPEC = 1e-4
LAM_SMOOTH = 1e-3

In [ ]:
import subprocess, sys
rc = subprocess.call([sys.executable, '-m', 'pip', 'install', '--quiet',
                       '--index-url', 'https://download.pytorch.org/whl/cu121',
                       'torch==2.4.1'])
print('pip install torch 2.4.1+cu121 exit code:', rc)

In [ ]:
import os, sys, subprocess, shutil, time
from pathlib import Path
INPUT = Path('/kaggle/input')
AUX_DIR = list(INPUT.rglob('combined_data.csv'))[0].parent
CODE_DIR = list(INPUT.rglob('scripts'))[0].parent
WORK = Path('/kaggle/working')
REPO_DIR = WORK / 'TaylorCouetteML'
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
shutil.copytree(CODE_DIR, REPO_DIR)
os.chdir(REPO_DIR)
INPUT_CSV = REPO_DIR / 'data' / 'Input' / 'combined_data.csv'
INPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(AUX_DIR / 'combined_data.csv', INPUT_CSV)
print('Setup ready, REPO_DIR =', REPO_DIR)

In [ ]:
OUT_DIR = WORK / 'runs' / 'cson_v1'
OUT_DIR.mkdir(parents=True, exist_ok=True)
args = [sys.executable, 'scripts/train_cson_pro.py',
        '--epochs', str(EPOCHS),
        '--batch', str(BATCH_SIZE),
        '--lr', str(LR),
        '--n_seeds', str(N_SEEDS),
        '--ctx_noise', str(CTX_NOISE),
        '--n_modes', str(N_MODES),
        '--d_model', str(D_MODEL),
        '--num_layers', str(NUM_LAYERS),
        '--lam_spec', str(LAM_SPEC),
        '--lam_smooth', str(LAM_SMOOTH),
        '--out_root', str(OUT_DIR)]
print('>>>', ' '.join(args))
t0 = time.time()
rc = subprocess.call(args, cwd=str(REPO_DIR))
print(f'<<< exit={rc} elapsed={(time.time()-t0)/60:.1f} min')
assert rc == 0, 'training failed'

In [ ]:
for root, dirs, files in os.walk(OUT_DIR):
    for f in files:
        p = Path(root) / f
        if p.suffix in ['.pt', '.pth', '.json', '.csv', '.npz']:
            print(p.relative_to(WORK), f'({p.stat().st_size/1e6:.2f} MB)')